# COMICS Panel Search — Query Playground

Developer-facing notebook for validating search quality and comparing dense, sparse, FTS, and combined retrieval over the Pinecone COMICS index.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone
from PIL import Image
import pandas as pd
from IPython.display import display, Markdown

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "comic-panels")
PINECONE_NAMESPACE = os.environ.get("PINECONE_NAMESPACE", "comics-v1")
COMICS_DATA_DIR = Path(os.environ.get("COMICS_DATA_DIR", "data/comics"))

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.preview.index(name=PINECONE_INDEX_NAME)

print("Connected to:", PINECONE_INDEX_NAME)
print("Namespace:   ", PINECONE_NAMESPACE)

In [ ]:
from src.search.search_dense import search_dense, search_dense_with_phrase_filter
from src.search.search_sparse import search_sparse
from src.search.search_fts import search_fts
from src.search.search_combined import search_combined
from src.search.router import route_query

In [ ]:
from src.embeddings.embed_images import embed_text_query
from src.embeddings.embed_sparse_text import embed_sparse_query


def embed_query_dense(query: str) -> list[float]:
    """Embed a text query using the text side of the OpenCLIP model."""
    return embed_text_query(query)


def embed_query_sparse(query: str) -> dict | None:
    """Embed a text query using pinecone-sparse-english-v0 in query mode."""
    return embed_sparse_query(query)

In [ ]:
def display_results(results, max_items=10, image_width=300):
    for i, hit in enumerate(results[:max_items], start=1):
        fields = hit.get("fields", hit)

        panel_id = hit.get("_id") or hit.get("panel_id")
        score = hit.get("_score") or hit.get("score") or hit.get("rrf_score")
        sources = hit.get("sources", fields.get("sources", []))

        image_path = fields.get("image_path") or hit.get("image_path")
        ocr_text = fields.get("ocr_text") or hit.get("ocr_text", "")
        comic_id = fields.get("comic_id") or hit.get("comic_id")
        page_num = fields.get("page_num") or hit.get("page_num")
        panel_num = fields.get("panel_num") or hit.get("panel_num")

        display(Markdown(f"### {i}. `{panel_id}`"))
        display(Markdown(
            f"**Score:** `{score}`  \n"
            f"**Sources:** `{sources}`  \n"
            f"**Comic:** `{comic_id}`  \n"
            f"**Page:** `{page_num}`  \n"
            f"**Panel:** `{panel_num}`"
        ))

        if image_path and Path(image_path).exists():
            img = Image.open(image_path)
            display(img.resize((image_width, int(image_width * img.height / img.width))))
        else:
            display(Markdown(f"`Image not found: {image_path}`"))

        if ocr_text:
            display(Markdown(f"**OCR:** {ocr_text}"))

        display(Markdown("---"))

## Dense Search

Semantic / visual search — best for descriptive scene queries.

In [ ]:
query = "a detective in a dark alley"
top_k = 10

dense_vector = embed_query_dense(query)

dense_response = search_dense(
    index=index,
    namespace=PINECONE_NAMESPACE,
    query_vector=dense_vector,
    top_k=top_k,
    filters={"is_ad_page": False},
)

dense_hits = dense_response["result"]["hits"]
display_results(dense_hits, max_items=top_k)

## Sparse Search

Keyword/lexical search — best for names, specific terms, and short phrases.

In [ ]:
query = "masked villain laboratory"
top_k = 10

sparse_vector = embed_query_sparse(query)

sparse_response = search_sparse(
    index=index,
    namespace=PINECONE_NAMESPACE,
    query_sparse=sparse_vector,
    top_k=top_k,
    filters={"is_ad_page": False},
)

sparse_hits = sparse_response["result"]["hits"]
display_results(sparse_hits, max_items=top_k)

## Full-Text Search

BM25 over OCR/dialogue — best for exact phrases, quoted dialogue, sound effects, Boolean operators.

In [ ]:
query = '"secret formula"'
top_k = 10

fts_response = search_fts(
    index=index,
    namespace=PINECONE_NAMESPACE,
    query=query,
    top_k=top_k,
    filters={"is_ad_page": False},
)

fts_hits = fts_response["result"]["hits"]
display_results(fts_hits, max_items=top_k)

## Combined Search (RRF)

Runs dense + sparse + FTS (based on router) and merges results with Reciprocal Rank Fusion.

In [ ]:
query = "a laboratory panel containing formula"
top_k = 10

route = route_query(query)
print("Route:", route)

dense_vector = embed_query_dense(query) if route["dense"] else None
sparse_vector = embed_query_sparse(query) if route["sparse"] else None

combined_hits = search_combined(
    index=index,
    namespace=PINECONE_NAMESPACE,
    query_text=query,
    dense_query_vector=dense_vector,
    sparse_query_vector=sparse_vector,
    top_k=top_k,
    filters={"is_ad_page": False},
    run_fts=route["fts"],
)

display_results(combined_hits, max_items=top_k)

## Compare Modes Side-by-Side

Run the same query across all three signals and compare results in a DataFrame.

In [ ]:
query = "secret formula laboratory"
top_k = 10

dense_vector = embed_query_dense(query)
sparse_vector = embed_query_sparse(query)

dense_hits = search_dense(
    index=index, namespace=PINECONE_NAMESPACE,
    query_vector=dense_vector, top_k=top_k,
    filters={"is_ad_page": False},
)["result"]["hits"]

sparse_hits = search_sparse(
    index=index, namespace=PINECONE_NAMESPACE,
    query_sparse=sparse_vector, top_k=top_k,
    filters={"is_ad_page": False},
)["result"]["hits"]

fts_hits = search_fts(
    index=index, namespace=PINECONE_NAMESPACE,
    query=query, top_k=top_k,
    filters={"is_ad_page": False},
)["result"]["hits"]


def hits_to_rows(mode, hits):
    rows = []
    for rank, hit in enumerate(hits, start=1):
        fields = hit.get("fields", {})
        rows.append({
            "mode": mode,
            "rank": rank,
            "panel_id": hit.get("_id"),
            "score": hit.get("_score"),
            "ocr_text": str(fields.get("ocr_text", ""))[:60],
            "comic_id": fields.get("comic_id"),
            "page_num": fields.get("page_num"),
            "panel_num": fields.get("panel_num"),
        })
    return rows


df = pd.DataFrame(
    hits_to_rows("dense", dense_hits)
    + hits_to_rows("sparse", sparse_hits)
    + hits_to_rows("fts", fts_hits)
)
df

## Metadata Filters

Examples of filtering by comic, page range, or ad-page exclusion.

In [ ]:
# Exclude ad pages (applied by default in all cells above)
filters = {"is_ad_page": False}

# Filter to a specific comic
# filters = {"comic_id": "some_comic_id", "is_ad_page": False}

# Filter to a page range
# filters = {"page_num": {"$gte": 5, "$lte": 20}, "is_ad_page": False}

query = "hero battles villain"
dense_vector = embed_query_dense(query)
response = search_dense(index=index, namespace=PINECONE_NAMESPACE,
                        query_vector=dense_vector, top_k=5, filters=filters)
display_results(response["result"]["hits"], max_items=5)

## Manual Evaluation

Run a battery of test queries across all modes and inspect results.

In [ ]:
test_queries = [
    '"secret formula"',
    "BANG",
    "a detective in a dark alley",
    "a superhero flying through the sky",
    "masked villain laboratory",
    "a laboratory panel containing formula",
]

for q in test_queries:
    display(Markdown(f"---\n### Query: `{q}`"))
    route = route_query(q)
    display(Markdown(f"Route: `{route}`"))

    dense_vector = embed_query_dense(q) if route["dense"] else None
    sparse_vector = embed_query_sparse(q) if route["sparse"] else None

    hits = search_combined(
        index=index,
        namespace=PINECONE_NAMESPACE,
        query_text=q,
        dense_query_vector=dense_vector,
        sparse_query_vector=sparse_vector,
        top_k=5,
        filters={"is_ad_page": False},
        run_fts=route["fts"],
    )

    display_results(hits, max_items=5)